In [ ]:
import importlib
import sys

spec = importlib.util.spec_from_file_location("lib", "lib/__init__.py")
module_obj = importlib.util.module_from_spec(spec)
sys.modules["lib"] = module_obj
spec.loader.exec_module(module_obj)

from lib import PipelineConfig

In [ ]:
if "snakemake" in locals():
    feeder_outputs = snakemake.params.feeder_outputs
    no_feeder_outputs = snakemake.params.no_feeder_outputs
    input_paths = snakemake.input
    pipeline_config = PipelineConfig(snakemake.params.config, **snakemake.params.pipeline_kwargs)
    scenario_name = snakemake.wildcards.scenario
    scenario_config = pipeline_config.scenarios[scenario_name]
else:
    raise Exception("This notebook is only snakemake-compatible for now")

In [ ]:
print("Values considered for the radius")
scenario_config.feeders.radiis

In [ ]:
print("Values considered for the frequency")
scenario_config.feeders.frequencies

In [ ]:
print("Values considered for the speeed")
scenario_config.feeders.speeds

In [ ]:
print("List of passed input paths")
feeder_outputs

Each Path is of the form /path/to/something/simulated_{scenario}_transitWithAbstractAccess_{radius}_{frequency}_{speed_index}
- radius: the value of the radius parameter
- frequency: value of the frequency
- speed_index: index of the speed value in the speeds list displayed above (I didn't want to risk passing floats around)

Next we display the path to the outputs of the simulation corresponding to the scenario without feeder service
The variable can be none if no such simulation is provided

In [ ]:
no_feeder_outputs

# What to do next ?

In general we want to measure the relationship between the level of introduction of feeder services and how much it is used and also the related PT offer. We also want to measure the source of the modal shift (whether it is mainly car trips that are replaced or other PT trips)

Some sanity checks, verify that all the scenarios have the same set of trips (person_id, person_trip_id, departure_time, origin, destination). If not, keep only the same ones while we investigate the source of the problem. Here we can have a few differences because some agents might get stuck.

In [ ]:
import pandas as pd
import os
import plotly.express as px
import re
import numpy as np
import math
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
# Load data

## No feeder

no_feeder_outputs = no_feeder_outputs[0]

nofeeder_legs = pd.read_csv(os.path.join(no_feeder_outputs, "eqasim_legs.csv"), sep=";")
nofeeder_pt = pd.read_csv(os.path.join(no_feeder_outputs, "eqasim_pt.csv"), sep=";")
nofeeder_trips = pd.read_csv(os.path.join(no_feeder_outputs, "eqasim_trips.csv"), sep=";")
nofeeder_routing_costs = pd.read_csv(os.path.join(no_feeder_outputs, "pt_routing_costs.csv"), sep=";")

# Feeders 

feeders_legs = [pd.read_csv(os.path.join(feeder_output, "eqasim_legs.csv"), sep=";") for feeder_output in feeder_outputs]
feeders_pt = [pd.read_csv(os.path.join(feeder_output, "eqasim_pt.csv"), sep=";") for feeder_output in feeder_outputs]
feeders_trips = [pd.read_csv(os.path.join(feeder_output, "eqasim_trips.csv"), sep=";") for feeder_output in feeder_outputs]
feeders_routing_costs = [pd.read_csv(os.path.join(feeder_output, "pt_routing_costs.csv"), sep=";") for feeder_output in feeder_outputs]

nofeeder_legs.head(5)

In [ ]:
## Extract parameters from file paths
def extract_parameters(path):
    """Extraire les paramètres des chemins de fichiers de scénarios feeder
    
    Pattern: feeder_{radius}_{frequency}_{speed_index}
    
    Args:
        path (str): Chemin vers le fichier de sortie du scénario
        
    Returns:
        tuple: (radius, frequency, speed, speed_index)
    """
    pattern = r'feeder_(\d+)_(\d+)_(\d+)'
    match = re.search(pattern, path)
    
    if match:
        radius = int(match.group(1))
        frequency = int(match.group(2))
        speed_index = int(match.group(3))
        speed = scenario_config.feeders.speeds[speed_index]
        return radius, frequency, speed, speed_index
    else:
        print(f"Attention: Impossible d'extraire les paramètres de {path}")
        return None, None, None, None

In [ ]:
print(f"Données chargées:")
print(f"- Scénario sans feeder: {len(nofeeder_routing_costs)} entrées de coût")
print(f"- Scénarios avec feeder: {len(feeders_routing_costs)} scénarios")
for i, feeder_data in enumerate(feeders_routing_costs):
    print(f"  Scénario {i}: {len(feeder_data)} entrées de coût")

In [ ]:
print("Colonnes disponibles dans nofeeder_routing_costs:")
print(list(nofeeder_routing_costs.columns))
print(f"Forme: {nofeeder_routing_costs.shape}")

print("\nAperçu des premières lignes:")
print(nofeeder_routing_costs.head())

print("\nTypes de données:")
print(nofeeder_routing_costs.dtypes)

if len(feeders_routing_costs) > 0:
    print(f"\nColonnes disponibles dans le premier scénario feeder:")
    print(list(feeders_routing_costs[0].columns))
    print(f"Forme: {feeders_routing_costs[0].shape}")

cost_columns = [col for col in nofeeder_routing_costs.columns if 'cost' in col.lower()]
print(f"\nColonnes contenant 'cost': {cost_columns}")

In [ ]:
# Sanity Check

key_columns = ["person_id", "person_trip_id"]
comparison_columns = ["origin_x", "origin_y", "destination_x", "destination_y", "departure_time"]
analysis_columns = key_columns + comparison_columns

summary = []

for i, feeder_trips in enumerate(feeders_trips):
    merged = nofeeder_trips[analysis_columns].merge(
        feeder_trips[analysis_columns],
        on=key_columns,
        suffixes=("_nofeeder", "_feeder"),
        how="outer",
        indicator=True
    )
    only_nofeeder = (merged['_merge'] == 'left_only').sum()
    only_feeder = (merged['_merge'] == 'right_only').sum()
    both = (merged['_merge'] == 'both').sum()
    pct_common = both / len(merged) * 100 if len(merged) > 0 else 0

    common = merged[merged['_merge'] == 'both']
    diffs = {}
    for col in comparison_columns:
        col_nofeeder = f"{col}_nofeeder"
        col_feeder = f"{col}_feeder"
        if col_nofeeder in common and col_feeder in common:
            mask = (~common[col_nofeeder].isna()) & (~common[col_feeder].isna())
            if col in ['origin_x', 'origin_y', 'destination_x', 'destination_y']:
                diff = (abs(common.loc[mask, col_nofeeder] - common.loc[mask, col_feeder]) > 1e-6).sum()
            else:
                diff = (common.loc[mask, col_nofeeder] != common.loc[mask, col_feeder]).sum()
            diffs[col] = diff


    summary.append({
        "scenario": i,
        "total": len(merged),
        "common": both,
        "only_nofeeder": only_nofeeder,
        "only_feeder": only_feeder,
        "pct_common": pct_common,
        "diffs": diffs,
    })

for s in summary:
    print(f"Scenario {s['scenario']}: {s['common']}/{s['total']} commun trips ({s['pct_common']:.1f}%)")
    if s['diffs']:
        diff_str = ", ".join(f"{k}:{v}" for k, v in s['diffs'].items() if v > 0)
        if diff_str:
            print(f"    Differences: {diff_str}")

df_summary = pd.DataFrame([{
    "Scenario": s["scenario"],
    "Common Trips": s["common"],
    "Only in NoFeeder": s["only_nofeeder"],
    "Only in Feeder": s["only_feeder"]
} for s in summary])

n_cols = 2
n_rows = math.ceil(len(summary) / n_cols)

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    specs=[[{'type':'domain'}]*n_cols for _ in range(n_rows)],
    subplot_titles=[f"Scenario {s['scenario']}" for s in summary],
)

for i, s in enumerate(summary):
    row = i // n_cols + 1
    col = i % n_cols + 1
    values = [s["common"], s["only_nofeeder"], s["only_feeder"]]
    labels = ["Common Trips", "Only in NoFeeder", "Only in Feeder"]
    fig.add_trace(
        go.Pie(
            labels=labels,
            values=values,
            name=f"Scenario {s['scenario']}",
            textinfo='percent',
            textposition='inside',
            insidetextorientation='radial'
        ),
        row=row, col=col
    )

fig.update_layout(
    title_text="Sanity Check - Répartition des Trips par Scénario",
    height=300*n_rows,
    width=700,
    showlegend=True
)

fig.show()


In [ ]:
def identify_cost_column(df):
    priority_columns = ['total_cost', 'routingCost', 'cost', 'totalCost', 'routing_cost']
    for col in priority_columns:
        if col in df.columns:
            return col
    cost_columns = [col for col in df.columns if 'cost' in col.lower()]
    if cost_columns:
        return cost_columns[0]
    numeric_columns = df.select_dtypes(include=['float64', 'int64']).columns
    if len(numeric_columns) > 0:
        return numeric_columns[0]
    raise ValueError("Impossible d'identifier une colonne de coût dans les données")

nofeeder_cost_col = identify_cost_column(nofeeder_routing_costs)
feeder_cost_cols = [identify_cost_column(df) for df in feeders_routing_costs]

def cost_stats(df, cost_column):
    cost_data = df[cost_column].dropna()
    if len(cost_data) == 0:
        return None
    Q1 = cost_data.quantile(0.25)
    Q3 = cost_data.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = cost_data[(cost_data < lower_bound) | (cost_data > upper_bound)]
    return {
        'count': len(cost_data),
        'mean': cost_data.mean(),
        'median': cost_data.median(),
        'std': cost_data.std(),
        'min': cost_data.min(),
        'max': cost_data.max(),
        'outliers(%)': len(outliers) / len(cost_data) * 100,
        'negatives': (cost_data < 0).sum(),
        'zeros': (cost_data == 0).sum()
    }

routing_stats = []
ref_stats = cost_stats(nofeeder_routing_costs, nofeeder_cost_col)
if ref_stats:
    ref_stats['scenario'] = "No Feeder"
    routing_stats.append(ref_stats)

for i, (feeder_routing_costs, cost_col) in enumerate(zip(feeders_routing_costs, feeder_cost_cols)):
    radius, frequency, speed, speed_index = extract_parameters(feeder_outputs[i])
    scenario_name = f"Feeder R{radius} F{frequency} S{speed:.1f}" if radius is not None else f"Feeder {i}"
    stats = cost_stats(feeder_routing_costs, cost_col)
    if stats:
        stats['scenario'] = scenario_name
        routing_stats.append(stats)

if routing_stats:
    df_routing_stats = pd.DataFrame(routing_stats).set_index('scenario')
    display(df_routing_stats.round(2))
else:
    print("Aucune statistique de coût n'a pu être calculée")

In [ ]:
merged_routing_costs = nofeeder_routing_costs.copy()
id_cols = ['person_id']
if 'trip_id' in merged_routing_costs.columns:
    id_cols.append('trip_id')
elif 'person_trip_id' in merged_routing_costs.columns:
    id_cols.append('person_trip_id')

cost_nofeeder = identify_cost_column(merged_routing_costs)
merged_routing_costs = merged_routing_costs.rename(columns={cost_nofeeder: 'routingCost_nofeeder'})

for i, feeder_df in enumerate(feeders_routing_costs):
    cost_feeder = identify_cost_column(feeder_df)
    merge_cols = [col for col in id_cols if col in feeder_df.columns]
    if not merge_cols:
        print(f"Feeder {i}: pas de colonnes communes")
        continue
    feeder_sub = feeder_df[merge_cols + [cost_feeder]].rename(columns={cost_feeder: f'routingCost_feeder_{i}'})
    merged_routing_costs = merged_routing_costs.merge(feeder_sub, on=merge_cols, how='left')

    diff_col = f'cost_difference_feeder_{i}'
    merged_routing_costs[diff_col] = merged_routing_costs['routingCost_nofeeder'] - merged_routing_costs[f'routingCost_feeder_{i}']
    diff = merged_routing_costs[diff_col].dropna()
    print(f"Feeder {i}: { (diff > 0).sum() } améliorations, { (diff < 0).sum() } dégradations")

print(f"Shape: {merged_routing_costs.shape}, Colonnes: {list(merged_routing_costs.columns)}")

In [ ]:
comparison_results = []

for i in range(len(feeders_routing_costs)):
    scenario_name = f"Feeder {i}"
    try:
        cost_nofeeder = merged_routing_costs['routingCost_nofeeder']
        cost_feeder = merged_routing_costs[f'routingCost_feeder_{i}']
        diff = merged_routing_costs[f'cost_difference_feeder_{i}']
        
        valid_mask = diff.notna()
        valid_nofeeder = cost_nofeeder[valid_mask]
        valid_diff = diff[valid_mask]
        valid_feeder = cost_feeder[valid_mask]

        improved = (valid_diff > 0).sum()
        worsened = (valid_diff < 0).sum()
        unchanged = (valid_diff == 0).sum()

        comparison_results.append({
        'scenario': scenario_name,
        'trips': len(valid_diff),
        'trips_improved': improved,
        'trips_worsened': worsened,
        'trips_unchanged': unchanged,
        'delta_mean_total': valid_diff.mean(),
        'delta_mean_diff': valid_diff[valid_diff != 0].mean(),
        'delta_total': valid_diff.sum(),
        'percent_improved': (improved / len(valid_diff)) * 100 if len(valid_diff) > 0 else 0
        })

    except Exception as e:
        print(f"{scenario_name}: Erreur - {e}")

if comparison_results:
    df_comparisons = pd.DataFrame(comparison_results)
    print(df_comparisons.round(2))
else:
    print("Aucune comparaison possible.")

In [ ]:
modes = set(nofeeder_trips["mode"].unique())
for feeder_trips in feeders_trips:
    modes = modes.union(set(feeder_trips["mode"].unique()))

pt_modes = set(nofeeder_pt["transit_mode"].unique())
for feeder_pt in feeders_pt:
    pt_modes = pt_modes.union(set(feeder_pt["transit_mode"].unique()))

In [ ]:
fact_nofeeder = 100
fact_feeder = 100

count_modes_nofeeder = {mode: len(nofeeder_trips[nofeeder_trips["mode"] == mode]) * fact_nofeeder for mode in modes}

count_modes_feeders = []
for i, feeder_trips in enumerate(feeders_trips):
    count_modes_feeder = {mode: len(feeder_trips[feeder_trips["mode"] == mode]) * fact_feeder for mode in modes}
    count_modes_feeders.append(count_modes_feeder)

all_data = []
for mode in modes:

    all_data.append({
        'mode': mode,
        'scenario': 'nofeeder',
        'count': count_modes_nofeeder[mode],
        'scenario_index': -1
    })
    
    for i, count_modes_feeder in enumerate(count_modes_feeders):
        all_data.append({
            'mode': mode,
            'scenario': f'feeder_{i}',
            'count': count_modes_feeder[mode],
            'scenario_index': i
        })

df_modes = pd.DataFrame(all_data)

df_modes.head(5)

In [ ]:
# Graphique 1: Comparaison des comptages par mode
fig1 = px.bar(df_modes, 
              x='mode', 
              y='count', 
              color='scenario',
              title='Comparaison des modes de transport par scénario',
              labels={'count': 'Nombre de trips', 'mode': 'Mode de transport'},
              barmode='group')
fig1.show()

# Graphique 2: Différences absolues pour chaque scénario feeder
diff_data = []
for i, count_modes_feeder in enumerate(count_modes_feeders):
    for mode in modes:
        diff = count_modes_feeder[mode] - count_modes_nofeeder[mode]
        diff_data.append({
            'mode': mode,
            'scenario': f'feeder_{i}',
            'diff_absolute': diff,
            'scenario_index': i
        })

df_diff = pd.DataFrame(diff_data)

fig2 = px.bar(df_diff, 
              x='mode', 
              y='diff_absolute', 
              color='scenario',
              title='Différences absolues (Feeder - NoFeeder)',
              labels={'diff_absolute': 'Différence (trips)', 'mode': 'Mode de transport'},
              barmode='group')
fig2.add_hline(y=0, line_dash="dash", line_color="black")
fig2.show()

# Graphique 3: Différences relatives en pourcentage
diff_percent_data = []
for i, count_modes_feeder in enumerate(count_modes_feeders):
    for mode in modes:
        if count_modes_nofeeder[mode] != 0:
            diff_pct = (count_modes_feeder[mode] - count_modes_nofeeder[mode]) / count_modes_nofeeder[mode] * 100
        else:
            diff_pct = float('inf') if count_modes_feeder[mode] > 0 else 0
        
        if diff_pct != float('inf'):
            diff_percent_data.append({
                'mode': mode,
                'scenario': f'feeder_{i}',
                'diff_percent': diff_pct,
                'scenario_index': i
            })

df_diff_pct = pd.DataFrame(diff_percent_data)

fig3 = px.bar(df_diff_pct, 
              x='mode', 
              y='diff_percent', 
              color='scenario',
              title='Différences relatives (%) - (Feeder - NoFeeder)/NoFeeder * 100',
              labels={'diff_percent': 'Différence (%)', 'mode': 'Mode de transport'},
              barmode='group')
fig3.add_hline(y=0, line_dash="dash", line_color="black")
fig3.show()

In [ ]:
count_pt_modes_nofeeder = {mode: len(nofeeder_pt[nofeeder_pt["transit_mode"] == mode]) * fact_nofeeder for mode in pt_modes}

count_pt_modes_feeders = []
for i, feeder_pt in enumerate(feeders_pt):
    count_pt_modes_feeder = {mode: len(feeder_pt[feeder_pt["transit_mode"] == mode]) * fact_feeder for mode in pt_modes}
    count_pt_modes_feeders.append(count_pt_modes_feeder)

all_pt_data = []
for mode in pt_modes:

    all_pt_data.append({
        'pt_mode': mode,
        'scenario': 'nofeeder',
        'count': count_pt_modes_nofeeder[mode],
        'scenario_index': -1
    })
    
    for i, count_pt_modes_feeder in enumerate(count_pt_modes_feeders):
        all_pt_data.append({
            'pt_mode': mode,
            'scenario': f'feeder_{i}',
            'count': count_pt_modes_feeder[mode],
            'scenario_index': i
        })

df_pt_modes = pd.DataFrame(all_pt_data)


In [ ]:

# Graphique 4: Comparaison des modes PT par scénario
fig4 = px.bar(df_pt_modes, 
              x='pt_mode', 
              y='count', 
              color='scenario',
              title='Comparaison des modes de transport public par scénario',
              labels={'count': 'Nombre de segments PT', 'pt_mode': 'Mode PT'},
              barmode='group')
fig4.show()

# Graphique 5: Différences absolues pour les modes PT
diff_pt_data = []
for i, count_pt_modes_feeder in enumerate(count_pt_modes_feeders):
    for mode in pt_modes:
        diff = count_pt_modes_feeder[mode] - count_pt_modes_nofeeder[mode]
        diff_pt_data.append({
            'pt_mode': mode,
            'scenario': f'feeder_{i}',
            'diff_absolute': diff,
            'scenario_index': i
        })

df_diff_pt = pd.DataFrame(diff_pt_data)

fig5 = px.bar(df_diff_pt, 
              x='pt_mode', 
              y='diff_absolute', 
              color='scenario',
              title='Différences absolues modes PT (Feeder - NoFeeder)',
              labels={'diff_absolute': 'Différence (segments PT)', 'pt_mode': 'Mode PT'},
              barmode='group')
fig5.add_hline(y=0, line_dash="dash", line_color="black")
fig5.show()